In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [8]:
file_path = 'train_SCkXwSx_cNdUk28_icK39vy_36Oa6WQ.csv'
df = pd.read_csv(file_path)
df.head()

,id,gender,area,qualification,income,marital_status,vintage,claim_amount,num_policies,policy,type_of_policy,cltv
0,1,Male,Urban,Bachelor,5L-10L,1,5,5790,More than 1,A,Platinum,64308
1,2,Male,Rural,High School,5L-10L,0,8,5080,More than 1,A,Platinum,515400
2,3,Male,Urban,Bachelor,5L-10L,1,8,2599,More than 1,A,Platinum,64212
3,4,Female,Rural,High School,5L-10L,0,7,0,More than 1,A,Platinum,97920
4,5,Male,Urban,High School,More than 10L,1,6,3508,More than 1,A,Gold,59736


## Exploring the Unique Values of all Columns

In [7]:
df.qualification.unique()

<StringArray>
['Bachelor', 'High School', 'Others']
Length: 3, dtype: str

In [9]:
X = df.drop(["id","cltv"],axis=1)
y = df["cltv"]

In [12]:
categorical_cols = X.select_dtypes(include=["object"]).columns
print(categorical_cols)

Index(['gender', 'area', 'qualification', 'income', 'num_policies', 'policy',
       'type_of_policy'],
      dtype='str')


C:\Users\TAMILARASU\AppData\Local\Temp\ipykernel_34932\3621530889.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns


In [16]:
## Machines Only Understand the Numbers not the Human Language so we need to convert into numbers.
for col in categorical_cols:
    X[col] = X[col].astype(str)
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

In [17]:
X.head()

,gender,area,qualification,income,marital_status,vintage,claim_amount,num_policies,policy,type_of_policy
0,1,1,0,1,1,5,5790,1,0,1
1,1,0,1,1,0,8,5080,1,0,1
2,1,1,0,1,1,8,2599,1,0,1
3,0,0,1,1,0,7,0,1,0,1
4,1,1,1,3,1,6,3508,1,0,0


In [34]:
import lightgbm as lgb
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

# 1. Load the test dataset
# (Make sure to replace 'test_...csv' with your actual test file name if it has a hash like your train file)
test_file_path = "test_opI5Wmb_N886n9d_eE68Ob0_dOXWldY.csv"  # or update to your exact test filename
test_df = pd.read_csv(test_file_path)

# Save the test 'id' column for the final submission file
test_ids = test_df["id"]

# 2. Prepare X_test by dropping 'id' and applying the same text-to-number encoding
X_test = test_df.drop(["id"], axis=1)

for col in categorical_cols:
  X_test[col] = X_test[col].astype(str)
  # Using the same label encoding approach
  le_test = LabelEncoder()
  X_test[col] = le_test.fit_transform(X_test[col])

# 3. Setup K-Fold Cross-Validation (5 splits)
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(df))
test_preds = np.zeros(len(test_df))

lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "random_state": 42,
    "verbose": -1,
}

print("Training LightGBM model across 5 folds...")

# 4. Training loop
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
  X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
  X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

  # Initialize and fit LightGBM
  model = lgb.LGBMRegressor(**lgb_params)
  model.fit(X_train, y_train)

  # Save validation predictions
  oof_preds[val_idx] = model.predict(X_val)

  # Predict on test data and average them across all folds
  test_preds += model.predict(X_test) / N_SPLITS

  # Print fold score
  fold_r2 = r2_score(y_val, oof_preds[val_idx])
  print(f"Fold {fold + 1} R2 Score: {fold_r2:.4f}")

# 5. Check Overall Out-Of-Fold R2 Score
overall_r2 = r2_score(y, oof_preds)
print(f"\n==========================================")
print(f"Overall Out-Of-Fold R2 Score: {overall_r2:.4f}")
print(f"==========================================")

# 6. Create Final Submission File
submission = pd.read_csv(
    "sample_submission_ku1VMOX_Gd11zvJ_VxdIEsN_3crCTYo.csv"
)  # Or your sample submission file
submission["cltv"] = test_preds
submission.to_csv("final_submission.csv", index=False)
print(
    "\nSuccess! 'final_submission.csv' has been generated and is ready for"
    " submission!"
)

Training LightGBM model across 5 folds...
Fold 1 R2 Score: 0.1507
Fold 2 R2 Score: 0.1505
Fold 3 R2 Score: 0.1465
Fold 4 R2 Score: 0.1589
Fold 5 R2 Score: 0.1496

Overall Out-Of-Fold R2 Score: 0.1512

Success! 'final_submission.csv' has been generated and is ready for submission!


In [37]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

# 1. Load test dataset using your exact filename
test_file_path = "test_opI5Wmb_N886n9d_eE68Ob0_dOXWldY.csv"
test_df = pd.read_csv(test_file_path)
test_ids = test_df["id"]

# 2. Prepare X and y from your main df dataframe
X = df.drop(["id", "cltv"], axis=1)
y = df["cltv"]

# Prepare X_test by dropping 'id'
X_test = test_df.drop(["id"], axis=1)

# 3. Handle Categorical Encoding safely on both Train and Test
categorical_cols = X.select_dtypes(include=["object"]).columns

for col in categorical_cols:
  X[col] = X[col].astype(str)
  X_test[col] = X_test[col].astype(str)

  le = LabelEncoder()
  combined = pd.concat([X[col], X_test[col]], axis=0)
  le.fit(combined)

  X[col] = le.transform(X[col])
  X_test[col] = le.transform(X_test[col])

# 4. Setup K-Fold Cross-Validation (5 splits)
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(df))
test_preds = np.zeros(len(test_df))

# 5. Fine-Tuned LightGBM Hyperparameters (optimized for high R2 performance)
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 7,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "random_state": 42,
    "verbose": -1,
}

print("Training Fine-Tuned LightGBM model across 5 folds...")

# 6. Training loop on original scale with early stopping
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
  X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
  X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

  model = lgb.LGBMRegressor(**lgb_params)
  model.fit(
      X_train,
      y_train,
      eval_set=[(X_val, y_val)],
      callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
  )

  # Predict on validation data
  oof_preds[val_idx] = model.predict(X_val)

  # Predict on test data and average across folds
  test_preds += model.predict(X_test) / N_SPLITS

  fold_r2 = r2_score(y_val, oof_preds[val_idx])
  print(f"Fold {fold + 1} Tuned R2 Score: {fold_r2:.4f}")

# 7. Check Overall Score
overall_r2 = r2_score(y, oof_preds)
print(f"\n==========================================")
print(f"Overall Tuned Out-Of-Fold R2 Score: {overall_r2:.4f}")
print(f"==========================================")

# 8. Create Final Submission File
submission = pd.read_csv("sample_submission_ku1VMOX_Gd11zvJ_VxdIEsN_3crCTYo.csv")
submission["cltv"] = test_preds
submission.to_csv("final_submission.csv", index=False)
print(
    "\nSuccess! Optimized 'final_submission.csv' generated and ready for"
    " submission!"
)

C:\Users\TAMILARASU\AppData\Local\Temp\ipykernel_34932\2230342230.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns
d:\Projects\CLTV-Insurance\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training Fine-Tuned LightGBM model across 5 folds...
Fold 1 Tuned R2 Score: 0.1589


d:\Projects\CLTV-Insurance\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 Tuned R2 Score: 0.1586


d:\Projects\CLTV-Insurance\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 3 Tuned R2 Score: 0.1570


d:\Projects\CLTV-Insurance\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4 Tuned R2 Score: 0.1654


d:\Projects\CLTV-Insurance\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 Tuned R2 Score: 0.1571

Overall Tuned Out-Of-Fold R2 Score: 0.1594

Success! Optimized 'final_submission.csv' generated and ready for submission!


In [38]:
# 1. Install XGBoost if you haven't already
# !uv pip install xgboost  # or !pip install xgboost

import xgboost as xgb
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

# 2. Setup storage for XGBoost predictions
xgb_oof_preds = np.zeros(len(df))
xgb_test_preds = np.zeros(len(test_df))

# XGBoost hyperparameter settings
xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "tree_method": "hist",
}

print("Training XGBoost model across 5 folds...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
  X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
  X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

  # Initialize and train XGBoost
  xgb_model = xgb.XGBRegressor(**xgb_params)
  xgb_model.fit(
      X_train,
      y_train,
      eval_set=[(X_val, y_val)],
      verbose=False,
  )

  xgb_oof_preds[val_idx] = xgb_model.predict(X_val)
  xgb_test_preds += xgb_model.predict(X_test) / N_SPLITS

  fold_r2 = r2_score(y_val, xgb_oof_preds[val_idx])
  print(f"XGBoost Fold {fold + 1} R2 Score: {fold_r2:.4f}")

# Overall XGBoost Score
xgb_overall_r2 = r2_score(y, xgb_oof_preds)
print(f"\nOverall XGBoost Out-Of-Fold R2 Score: {xgb_overall_r2:.4f}")

# 3. Blending LightGBM and XGBoost Predictions (Ensembling for Maximum Accuracy)
# Give slightly more weight to LightGBM or blend them 50-50
final_blended_oof = (0.5 * oof_preds) + (0.5 * xgb_oof_preds)
final_blended_test = (0.5 * test_preds) + (0.5 * xgb_test_preds)

blended_overall_r2 = r2_score(y, final_blended_oof)
print(f"\n==========================================")
print(
    f"🔥 Blended Ensemble (LGBM + XGB) Overall R2 Score:"
    f" {blended_overall_r2:.4f}"
)
print(f"==========================================")

# 4. Save the ultimate submission file
submission = pd.read_csv("sample_submission_ku1VMOX_Gd11zvJ_VxdIEsN_3crCTYo.csv")
submission["cltv"] = final_blended_test
submission.to_csv("final_submission.csv", index=False)
print(
    "\nSuccess! Ultimate blended 'final_submission.csv' generated and ready"
    " for top leaderboard ranking!"
)

Training XGBoost model across 5 folds...
XGBoost Fold 1 R2 Score: 0.1437
XGBoost Fold 2 R2 Score: 0.1430
XGBoost Fold 3 R2 Score: 0.1407
XGBoost Fold 4 R2 Score: 0.1534
XGBoost Fold 5 R2 Score: 0.1461

Overall XGBoost Out-Of-Fold R2 Score: 0.1454

🔥 Blended Ensemble (LGBM + XGB) Overall R2 Score: 0.1559

Success! Ultimate blended 'final_submission.csv' generated and ready for top leaderboard ranking!
